# 05c · Offset sweeps (inference only, ≈ 1–2 h)
Run **after notebooks 05a and 05b have both finished** (any GPU). Writes `sweep_val.csv` / `sweep_test.csv` into each LOIO run folder and `sweep_test.csv` into each region-split run folder. Resumable.

In [ ]:
# --- Setup (run every session) ---
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/gwfss_stem_seg'
DATA_ROOT = '/content/gwfss'                      # local disk: fast I/O, rebuilt each session
SPLIT_FILE = f'{PROJECT}/splits/splits.json'      # frozen once, shared by all runs
RESULTS = f'{PROJECT}/results'                    # checkpoints, logs, metrics (on Drive)

import sys, os
sys.path.insert(0, f'{PROJECT}/src')
!pip -q install segmentation-models-pytorch==0.5.0
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from gwfss_stem import data
manifest = data.prepare_dataset(DATA_ROOT)   # downloads GWFSS_v1.0 from Hugging Face (~600 MB) if not present
print(len(manifest), 'images')

In [ ]:
from gwfss_stem import plan, engine
import pandas as pd
import glob
loio = [os.path.join(RESULTS, engine.run_name({**engine.DEFAULTS, **r})) for r in plan.loio_plan()]
missing = [d for d in loio if not os.path.exists(f'{d}/final.pt')]
print(len(loio) - len(missing), 'of', len(loio), 'LOIO runs finished')
region = sorted(os.path.dirname(p) for p in glob.glob(f'{RESULTS}/region/*/*/scale1.0/*/final.pt'))
print(len(region), 'region runs (expected 36)')
for d in loio:
    if d in missing: continue
    engine.offset_sweep(d, DATA_ROOT, SPLIT_FILE, subset='val')
    engine.offset_sweep(d, DATA_ROOT, SPLIT_FILE, subset='test')
for d in region:
    engine.offset_sweep(d, DATA_ROOT, SPLIT_FILE, subset='test')
print('done')